In [3]:
import pandas as pd
import numpy as np

# 1️⃣ Load your groundwater dataset
gw_data = pd.read_excel(r'D:\Thesis\Main\Data\GW table\Weekly_Groundwater_2023_Spline.xlsx')  # change path if needed

# 2️⃣ Identify weekly groundwater level columns (assuming they start with 'Week_')
week_cols = [col for col in gw_data.columns if col.startswith('Week_')]

# 3️⃣ Calculate statistics for each station
gw_data['Max_Level'] = gw_data[week_cols].max(axis=1)
gw_data['Min_Level'] = gw_data[week_cols].min(axis=1)
gw_data['Mean_Level'] = gw_data[week_cols].mean(axis=1)
gw_data['Range'] = gw_data['Max_Level'] - gw_data['Min_Level']

# 4️⃣ Calculate Recharge Index
gw_data['Recharge_Index'] = gw_data['Range'] / gw_data['Mean_Level']

# 5️⃣ Classify recharge potential
def classify_recharge(index):
    if index > 0.3:
        return 3  # High
    elif index > 0.15:
        return 2  # Moderate
    else:
        return 1  # Low

gw_data['Recharge_Class'] = gw_data['Recharge_Index'].apply(classify_recharge)

# 6️⃣ Save processed recharge labels to CSV
gw_data[['Station', 'Latitude', 'Longitude', 'Recharge_Index', 'Recharge_Class']].to_excel(
    r'D:\Thesis\Main\Data\GW table\Weekly_Groundwater_2023_Final_Filled.xlsx', index=False
)

print("Recharge classification completed!")
print(gw_data[['Station', 'Recharge_Index', 'Recharge_Class']].head())


Recharge classification completed!
         Station  Recharge_Index  Recharge_Class
0  GT8110006 gww        0.907920               3
1  GT8110012 gww        0.682431               3
2  GT8110502 gww        0.773311               3
3  GT8112002 gww        0.509744               3
4  GT8112003 gww        1.641751               3


In [3]:
import pandas as pd
import numpy as np

# 1️⃣ Load groundwater dataset
gw_data = pd.read_excel(
    r'D:\Thesis\Main\Data\GW table\Weekly_Groundwater_2023_Spline.xlsx'
)

# 2️⃣ Identify weekly groundwater level columns
week_cols = [col for col in gw_data.columns if col.startswith('Week_')]

# 3️⃣ Calculate groundwater statistics
gw_data['Max_Level'] = gw_data[week_cols].max(axis=1)
gw_data['Min_Level'] = gw_data[week_cols].min(axis=1)
gw_data['Mean_Level'] = gw_data[week_cols].mean(axis=1)
gw_data['Range'] = gw_data['Max_Level'] - gw_data['Min_Level']

# 4️⃣ Calculate Recharge Index
gw_data['Recharge_Index'] = gw_data['Range'] / gw_data['Mean_Level']


def classify_recharge_binary(index):
    if index > 0.4:
        return 1  # Recharge
    else:
        return 0  # No Recharge

gw_data['Recharge_Class'] = gw_data['Recharge_Index'].apply(classify_recharge_binary)

# 6️⃣ Save processed recharge labels
gw_data[['Station', 'Latitude', 'Longitude', 'Recharge_Index', 'Recharge_Class']].to_excel(
    r'D:\Thesis\Main\Data\GW table\Dataset_binary1.xlsx',
    index=False
)

# 7️⃣ Quick check
print("Binary recharge classification completed!\n")
print(gw_data['Recharge_Class'].value_counts())
print(gw_data[['Station', 'Recharge_Index', 'Recharge_Class']].head())


Binary recharge classification completed!

Recharge_Class
1    37
0    14
Name: count, dtype: int64
         Station  Recharge_Index  Recharge_Class
0  GT8110006 gww        0.907920               1
1  GT8110012 gww        0.682431               1
2  GT8110502 gww        0.773311               1
3  GT8112002 gww        0.509744               1
4  GT8112003 gww        1.641751               1


In [2]:
pip install rasterio geopandas openpyxl


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
   - -------------------------------------- 1.0/22.9 MB 19.5 MB/s eta 0:00:02
   ------- -------------------------------- 4.2/22.9 MB 10.7 MB/s eta 0:00:02
   ------- -------------------------------- 4.2/22.9 MB 10.7 MB/s eta 0:00:02
   ----------- ---------------------------- 6.8/22.9 MB 8.0 MB/s eta 0:00:03
   ------------------- -------------------- 11.3/22.9 MB 10.7 MB/s eta 0:00:02
   ------------------------ --------------- 13.9/22.9 MB 11.2 MB/s eta 0:00:01
   ---------------------------- ----------- 16.5/22.9 MB 11.3 MB/s eta 0:00:01
   --------------------------------- ------ 19.1/22.9 MB 11.5 MB/s eta 0:00:01
   ------------------------------------- -- 21.5/22.9 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 22.9/22.9 MB 11.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:-

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import Point

# ====================================================
# 1️⃣ Load groundwater station Excel file
# ====================================================
gw_data = pd.read_excel(
    r'D:\Thesis\Main\Data\GW table\Weekly_Groundwater_2023_Spline.xlsx'
)

# Ensure correct column names
# Must contain: Latitude, Longitude
print("Columns in file:\n", gw_data.columns)

# ====================================================
# 2️⃣ Convert to GeoDataFrame
# ====================================================
geometry = [Point(xy) for xy in zip(gw_data['Longitude'], gw_data['Latitude'])]
gdf = gpd.GeoDataFrame(gw_data, geometry=geometry, crs="EPSG:4326")

# ====================================================
# 3️⃣ Function to extract raster values
# ====================================================
def extract_raster_values(raster_path, gdf, column_name):
    
    with rasterio.open(raster_path) as src:
        
        # Reproject points if CRS mismatch
        if gdf.crs != src.crs:
            gdf_proj = gdf.to_crs(src.crs)
        else:
            gdf_proj = gdf
        
        coords = [(x,y) for x,y in zip(
            gdf_proj.geometry.x,
            gdf_proj.geometry.y
        )]
        
        values = [val[0] for val in src.sample(coords)]
    
    gdf[column_name] = values
    return gdf

# ====================================================
# 4️⃣ Extract categorical raster values
# ====================================================

gdf = extract_raster_values(
    r'D:\Thesis\Main\Infulencing factors\LULC\Final\lulc_pro_30.tif',
    gdf,
    'LULC'
)

gdf = extract_raster_values(
    r'D:\Thesis\Main\Infulencing factors\Geology\Geology_ras.tif',
    gdf,
    'Geology'
)

gdf = extract_raster_values(
    r'D:\Thesis\Main\Infulencing factors\Soil\SOil_ras.tif',
    gdf,
    'Soil'
)

print("Raster extraction completed.")

# ====================================================
# 5️⃣ Calculate Recharge Index
# ====================================================
week_cols = [col for col in gdf.columns if col.startswith('Week_')]

gdf['Max_Level'] = gdf[week_cols].max(axis=1)
gdf['Min_Level'] = gdf[week_cols].min(axis=1)
gdf['Mean_Level'] = gdf[week_cols].mean(axis=1)
gdf['Range'] = gdf['Max_Level'] - gdf['Min_Level']

gdf['Recharge_Index'] = gdf['Range'] / gdf['Mean_Level']

def classify_recharge_binary(index):
    return 1 if index > 0.4 else 0

gdf['Recharge_Class'] = gdf['Recharge_Index'].apply(classify_recharge_binary)

# ====================================================
# 6️⃣ One-Hot Encoding
# ====================================================
categorical_columns = ['LULC', 'Geology', 'Soil']

gdf_encoded = pd.get_dummies(
    gdf,
    columns=categorical_columns,
    drop_first=False
)

# ====================================================
# 7️⃣ Remove unnecessary columns
# ====================================================
columns_to_drop = week_cols + [
    'Max_Level', 'Min_Level', 'Mean_Level', 'Range',
    'geometry'
]

gdf_encoded = gdf_encoded.drop(columns=columns_to_drop)

# ====================================================
# 8️⃣ Export Final ML Dataset
# ====================================================
gdf_encoded.to_excel(
    r'D:\Thesis\Main\Data\GW table\Final_ML_Dataset_OHE.xlsx',
    index=False
)

print("\nFinal ML-ready dataset created successfully!")
print(gdf_encoded.head())

Columns in file:
 Index(['Station', 'Latitude', 'Longitude', 'Week_1', 'Week_2', 'Week_3',
       'Week_4', 'Week_5', 'Week_6', 'Week_7', 'Week_8', 'Week_9', 'Week_10',
       'Week_11', 'Week_12', 'Week_13', 'Week_14', 'Week_15', 'Week_16',
       'Week_17', 'Week_18', 'Week_19', 'Week_20', 'Week_21', 'Week_22',
       'Week_23', 'Week_24', 'Week_25', 'Week_26', 'Week_27', 'Week_28',
       'Week_29', 'Week_30', 'Week_31', 'Week_32', 'Week_33', 'Week_34',
       'Week_35', 'Week_36', 'Week_37', 'Week_38', 'Week_39', 'Week_40',
       'Week_41', 'Week_42', 'Week_43', 'Week_44', 'Week_45', 'Week_46',
       'Week_47', 'Week_48', 'Week_49', 'Week_50', 'Week_51', 'Week_52'],
      dtype='object')
Raster extraction completed.

Final ML-ready dataset created successfully!
         Station  Latitude  Longitude  Recharge_Index  Recharge_Class  LULC_1  \
0  GT8110006 gww   24.1973    88.8380        0.907920               1   False   
1  GT8110012 gww   24.1851    88.8397        0.682431       

In [1]:
import pandas as pd
import numpy as np

# 1️⃣ Load groundwater dataset
gw_data = pd.read_excel(
    r'D:\Thesis\Main\Data\GW table\Weekly_Groundwater_2023_Spline.xlsx'
)

# 2️⃣ Identify weekly groundwater level columns
week_cols = [col for col in gw_data.columns if col.startswith('Week_')]

# 3️⃣ Calculate groundwater statistics
gw_data['Max_Level'] = gw_data[week_cols].max(axis=1)
gw_data['Min_Level'] = gw_data[week_cols].min(axis=1)
gw_data['Mean_Level'] = gw_data[week_cols].mean(axis=1)
gw_data['Range'] = gw_data['Max_Level'] - gw_data['Min_Level']

# 4️⃣ Calculate Recharge Index
gw_data['Recharge_Index'] = gw_data['Range'] / gw_data['Mean_Level']


def classify_recharge_binary(index):
    if index > 0.6:
        return 1  # Recharge
    else:
        return 0  # No Recharge

gw_data['Recharge_Class'] = gw_data['Recharge_Index'].apply(classify_recharge_binary)

# 6️⃣ Save processed recharge labels
gw_data[['Station', 'Latitude', 'Longitude', 'Recharge_Index', 'Recharge_Class']].to_excel(
    r'D:\Thesis\Main\Data\GW table\Dataset_binary11.xlsx',
    index=False
)

# 7️⃣ Quick check
print("Binary recharge classification completed!\n")
print(gw_data['Recharge_Class'].value_counts())
print(gw_data[['Station', 'Recharge_Index', 'Recharge_Class']].head())


Binary recharge classification completed!

Recharge_Class
1    26
0    25
Name: count, dtype: int64
         Station  Recharge_Index  Recharge_Class
0  GT8110006 gww        0.907920               1
1  GT8110012 gww        0.682431               1
2  GT8110502 gww        0.773311               1
3  GT8112002 gww        0.509744               0
4  GT8112003 gww        1.641751               1
